# 🚀 Session 6 — The Ultimate AI Game (Student)

Welcome to the Grand Finale! 🎉

So far, you’ve built an AI that can recognize images and a game loop. But today, we elevate it to the next level!

Today, we will:
1. **The Real Brain**: Replace the file-name cheating with your actual Neural Network from Session 3.
2. **The Dynamic Soul**: Use a real Large Language Model (LLM) to give your AI a dynamic personality.
3. **Give It Voice**: Make the AI actually SPEAK to you using Text-to-Speech.
4. **The Arcade Interface**: Put everything into a beautiful Web UI using Gradio!


In [ ]:
# 📦 Let's import all our superpowers in one place!
import json
import random
from pathlib import Path

# Superpower 1: AI Vision
import torch
import torch.nn as nn
from PIL import Image
from torchvision import models, transforms

# Superpower 2: LLM Brains (Gemini)
import google.generativeai as genai

# Superpower 3: Voice AI
from gtts import gTTS
import librosa

# Superpower 4: Web UI
import gradio as gr

print("✅ Superpowers loaded!")

## Step 1 — Load Your Game Memory
Let's load what you saved in previous sessions.

In [ ]:
settings_path = Path("session_data/game_settings.json")

if settings_path.exists():
    with open(settings_path, "r", encoding="utf-8") as f:
        settings = json.load(f)
else:
    settings = {"classes": ["cat", "dog"], "num_classes": 2, "personality": "sassy"}

classes = settings["classes"]
num_classes = settings.get("num_classes", len(classes))
personality = settings.get("personality", "sassy")

print("Labels:", classes)
print("Personality to use:", personality)

## Step 2 — The Real AI Brain (Vision)
Load your `ResNet18` model from Session 3. If you don't have one, it will load the default.

In [ ]:
model_path = Path("models/classifier.pt")
model = models.resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, num_classes)

if model_path.exists():
    state = torch.load(model_path, map_location="cpu")
    model.load_state_dict(state)
    print("✅ Real AI Brain Loaded!")
else:
    print("⚠️ No trained model found, using random guesses.")

model.eval()

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

def predict_image(image):
    """Takes a PIL Image and returns the label name."""
    image = image.convert("RGB")
    x = transform(image).unsqueeze(0)
    with torch.no_grad():
        logits = model(x)
        probs = torch.softmax(logits, dim=1)[0]
    best_idx = int(torch.argmax(probs).item())
    return classes[best_idx]

## Step 3 — The Dynamic Soul (Prompt Engineering)
Instead of a boring list of responses, we connect to Google Gemini to write unique, creative responses on the fly!
- Insert your API key provided by your teacher.

In [ ]:
# Paste your API key here 🔑
GOOGLE_API_KEY = "YOUR_API_KEY_HERE"

genai.configure(api_key=GOOGLE_API_KEY)

# Create the Gemini AI Model instance
try:
    chat_model = genai.GenerativeModel('gemini-1.5-flash')
except:
    chat_model = None
    print("You need to put in your API key!")

In [ ]:
def get_ai_reaction(is_correct, ai_guess, player_guess, game_personality):
    """
    Uses Gemini to generate a fun, creative response!
    """
    if GOOGLE_API_KEY == "YOUR_API_KEY_HERE":
        return f"{game_personality.upper()}: You need an API key to talk to me!"
        
    status = "correct" if is_correct else "wrong"
    
    # This is Prompt Engineering!
    prompt = f"""
    You are a game show host with a {game_personality} personality.
    The player just guessed '{player_guess}'. The true answer was '{ai_guess}'.
    The player was {status}.
    Keep it entirely short (under 2 sentences) and react to their guess with a {game_personality} vibe.
    """
    
    response = chat_model.generate_content(prompt)
    return response.text.replace('*', '').strip()

## Step 4 — Voice AI (Text-to-Speech)
Let's turn the LLM text into spoken audio using `gTTS`.

In [ ]:
import tempfile
import os

def speak_text(text):
    """Takes text and returns a path to an audio file."""
    tts = gTTS(text, lang='en', tld='com') # Try 'co.uk' for British accent!
    # Create a temporary file path to store our sound
    audio_path = tempfile.mktemp(suffix=".mp3")
    tts.save(audio_path)
    return audio_path

## Step 5 — The Arcade Interface (Gradio!)
We tie the Brain, Soul, and Voice into one incredible Web User Interface.
No more boring text loops. Player uploads an image, guesses, and experiences the WOW!

In [ ]:
score = 0

def play_round(image, player_guess):
    global score
    
    if image is None:
        return "Upload an image first!", None, f"Score: {score}"
    
    # 1. BRAIN: Predict the image
    ai_guess = predict_image(image)
    is_correct = (player_guess.strip().lower() == ai_guess.lower())
    
    # Score Math
    if is_correct:
        score += 10
    else:
        score -= 5
        
    # 2. SOUL: Get fun LLM response
    text_response = get_ai_reaction(is_correct, ai_guess, player_guess, personality)
    
    # 3. VOICE: Speak it
    audio_response = speak_text(text_response)
    
    if is_correct:
        final_text = f"🎉 {text_response}"
    else:
        final_text = f"❌ {text_response}\n(The AI thought it was a {ai_guess})"
        
    return final_text, audio_response, f"Score: {score}"

print("✅ Game Engine Ready!")

## 🚀 LAUNCH THE GAME!
Run this cell and play your game directly below!

In [ ]:
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown(f"# 🤖 {personality.capitalize()} AI Arcade")
    gr.Markdown("Upload an image, guess what it is, and see if the AI agrees!")
    
    with gr.Row():
        with gr.Column():
            image_input = gr.Image(type="pil", label="1. Drag & Drop Image")
            guess_input = gr.Textbox(label="2. Type your guess (e.g. cat)")
            submit_btn = gr.Button("🕹️ Play Turn", variant="primary")
        
        with gr.Column():
            score_display = gr.Markdown("### Score: 0")
            text_output = gr.Textbox(label="AI Chat", lines=3)
            audio_output = gr.Audio(label="AI Voice", autoplay=True)
            
    submit_btn.click(
        fn=play_round,
        inputs=[image_input, guess_input],
        outputs=[text_output, audio_output, score_display]
    )

demo.launch(debug=True, share=False)

## 🌟 Course Wrap-up
Congratulations! You created a multi-modal AI application!

- You used **Computer Vision** to see (ResNet).
- You used a **Large Language Model** to think & talk (Gemini).
- You used **Audio Generation** to speak (gTTS).
- You used a **Graphic User Interface** (Gradio) to pull it all together.

**You are now an AI Developer!** 🎓